# Финальная проверка выбранной модели

Цель — оценить логистическую регрессию v0 на отдельной тестовой
выборке и сравнить её с правилом PAY_0 при нагрузке 10% и 20%.

Модель, признаки, гиперпараметры и порядок отбора зафиксированы
до открытия test.csv.

После просмотра результатов изменения модели, признаков и правил
не допускаются.

In [1]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

TARGET = "default payment next month"
RANDOM_STATE = 42
TOP_SHARES = [0.10, 0.20]

In [2]:
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train.shape, test.shape

((24000, 25), (6000, 25))

In [4]:
data_summary = pd.DataFrame({
    "Выборка": ["train", "test"],
    "Клиентов": [len(train), len(test)],
    "Событий": [train[TARGET].sum(), test[TARGET].sum()],
    "Доля событий, %": [
        train[TARGET].mean() * 100,
        test[TARGET].mean() * 100
    ]
})

data_summary.round(2)

,Выборка,Клиентов,Событий,"Доля событий, %"
0,train,24000,5309,22.12
1,test,6000,1327,22.12


In [5]:
payment_status_columns = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

for data in [train, test]:
    data["TOTAL_BILL_AMT"] = data[bill_columns].sum(axis=1)
    data["TOTAL_PAY_AMT"] = data[payment_columns].sum(axis=1)
    data["NUM_DELINQ"] = (
        data[payment_status_columns] > 0
    ).sum(axis=1)

In [6]:
feature_columns = [
    "LIMIT_BAL",
    "AGE",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    *payment_status_columns,
    *bill_columns,
    *payment_columns,
    "TOTAL_BILL_AMT",
    "TOTAL_PAY_AMT",
    "NUM_DELINQ"
]

X_train = train[feature_columns]
y_train = train[TARGET]

X_test = test[feature_columns]
y_test = test[TARGET]

X_train.shape, X_test.shape

((24000, 26), (6000, 26))

In [7]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

numeric_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

final_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "logistic_regression",
            LogisticRegression(
                C=1.0,
                max_iter=2000,
                solver="lbfgs"
            )
        )
    ]
)

final_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['LIMIT_BAL', 'AGE',
                                                   'BILL_AMT1', 'BILL_AMT2',
                                                   'BILL_AMT3', 'BILL_AMT4',
                                                   'BILL_AMT5', 'BILL_AMT6',
                                                   'PAY_AMT1', 'PAY_AMT2',
                                                   'PAY_AMT3', 'PAY_AMT4',
                                                   'PAY_AMT5', 'PAY_AMT6',
                                                   'TOTAL_BILL_AMT',
                                                   'TOTAL_PAY_AMT',
                                                   'NUM_DELINQ']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['SEX', 'EDUCATION',
                                                   'MARRIAGE', 'PAY_0', 'PAY_2',
                                                   'PAY_3', 'PAY_4', 'PAY_5',
                                                   'PAY_6'])])),
                ('logistic_regression', LogisticRegression(max_iter=2000))])

In [8]:
test_scores = final_model.predict_proba(X_test)[:, 1]

test_predictions = test[["ID", TARGET, "PAY_0"]].copy()
test_predictions["model_score"] = test_scores

test_predictions.head(10)

,ID,default payment next month,PAY_0,model_score
0,6908,0,-1,0.155864
1,24576,0,-1,0.197602
2,26767,0,0,0.140424
3,2157,1,0,0.096311
4,3180,0,-2,0.068930
5,29383,0,1,0.272113
6,24455,0,0,0.114755
7,23887,0,0,0.073657
8,5505,0,-1,0.094654
9,29721,0,0,0.111122


In [9]:
def evaluate_top_share(
    data,
    target_column,
    scores,
    share,
    method_name
):
    selected_count = int(np.ceil(len(data) * share))

    ranked = (
        data
        .assign(score=scores)
        .sort_values(
            ["score", "ID"],
            ascending=[False, True]
        )
        .head(selected_count)
    )

    y = ranked[target_column]
    base_rate = data[target_column].mean()

    return {
        "Метод": method_name,
        "Нагрузка, %": len(ranked) / len(data) * 100,
        "Событий в отборе": int(y.sum()),
        "Доля событий, %": y.mean() * 100,
        "Recall, %": y.sum() / data[target_column].sum() * 100,
        "Lift": y.mean() / base_rate
    }

In [10]:
final_results = []

for share in TOP_SHARES:
    rule_result = evaluate_top_share(
        test_predictions,
        TARGET,
        test_predictions["PAY_0"].astype(float),
        share=share,
        method_name="Правило PAY_0"
    )

    model_result = evaluate_top_share(
        test_predictions,
        TARGET,
        test_predictions["model_score"],
        share=share,
        method_name="Логистическая регрессия v0"
    )

    final_results.append(rule_result)
    final_results.append(model_result)

final_results_df = pd.DataFrame(final_results)

final_results_df.round(2)

,Метод,"Нагрузка, %",Событий в отборе,"Доля событий, %","Recall, %",Lift
0,Правило PAY_0,10.0,417,69.50,31.42,3.14
1,Логистическая регрессия v0,10.0,407,67.83,30.67,3.07
2,Правило PAY_0,20.0,618,51.50,46.57,2.33
3,Логистическая регрессия v0,20.0,652,54.33,49.13,2.46


## Финальный результат

Модель и правило PAY_0 оценены на отдельной тестовой выборке
из 6 000 клиентов при нагрузке 10% и 20%.

Это единственная финальная проверка выбранного пайплайна.
После просмотра результатов модель, признаки, гиперпараметры
и правила отбора не изменяются.

## Финальный результат

Логистическая регрессия v0 и правило PAY_0 оценены на отдельной
тестовой выборке из 6 000 клиентов.

При нагрузке 10% правило PAY_0 охватило 417 событий из 1 327,
а модель v0 — 407 событий. Правило показало более высокий recall
(31,42% против 30,67%) и lift (3,14 против 3,07).

При нагрузке 20% модель v0 охватила 652 события против 618
у правила. Модель показала более высокий recall (49,13% против
46,57%) и lift (2,46 против 2,33).

Результат показывает, что при узком лимите ручной проверки
простое правило PAY_0 остаётся конкурентоспособным. При более
широком лимите 20% логистическая регрессия v0 обеспечивает
больший охват событий.

Результат получен на отдельной тестовой выборке. Модель, признаки,
гиперпараметры и порядок отбора были зафиксированы до проверки.
После просмотра теста дальнейшие изменения не вносятся.